> Experiments with [xsa.access](../trapallada/xsa/access.py) module

In [1]:
import pathlib
import os
import time

# import pandas as pd

import trapallada.xsa.access
import trapallada.xsa.util
import trapallada.plot

from astroquery.esa.xmm_newton import XMMNewton
# import astropy.io
from astropy.table import Table

import altair as alt

So that the notebook is aware of modifications in the the `import`ed modules

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


All the available tables (only a few are printed)

In [3]:
tables = trapallada.xsa.access.get_all_tables()
tables[:5]

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]


['public.dual',
 'tap_config.coord_sys',
 'tap_config.properties',
 'tap_schema.columns',
 'tap_schema.key_columns']

Get *metadata* for observations whose duration is in a given `duration_interval`

In [4]:
obs_metadata = trapallada.xsa.access.get_observations_in_interval(duration_interval=(35_000, 45_000))
obs_metadata

number of chunks: 1


,observation_id,duration
0,0006010301,35210
1,0006010401,35915
2,0008220201,42610
3,0009650101,41415
4,0009650201,36590
...,...,...
1430,0861950231,38363
1431,0861950234,35021
1432,0932390701,43000
1433,0932391501,37500


The directory where observations' data will be saved

In [5]:
output_directory =  pathlib.Path(os.environ['MY_DATA_DIR']) / 'trapallada'
output_directory.mkdir(exist_ok=True)

Let us select a few observations to download

In [6]:
obs_to_download = obs_metadata['observation_id'].iloc[:10]
obs_to_download

0    0006010301
1    0006010401
2    0008220201
3    0009650101
4    0009650201
5    0011420101
6    0011420201
7    0011430201
8    0011450201
9    0011830201
Name: observation_id, dtype: object

We download the first few observations

In [7]:
for obs_id in obs_to_download:

    print(f'Requesting observation {obs_id}')
    
    trapallada.xsa.access.download_extract_filter(obs=obs_id, output_directory=output_directory)
    
    # in order to, hopefully, avoid bans/errors...
    time.sleep(5)

Requesting observation 0006010301
found directory "/home/manu/data/trapallada/0006010301", skipping...
Requesting observation 0006010401
found directory "/home/manu/data/trapallada/0006010401", skipping...
Requesting observation 0008220201
found directory "/home/manu/data/trapallada/0008220201", skipping...
Requesting observation 0009650101
found directory "/home/manu/data/trapallada/0009650101", skipping...
Requesting observation 0009650201
found directory "/home/manu/data/trapallada/0009650201", skipping...
Requesting observation 0011420101
found directory "/home/manu/data/trapallada/0011420101", skipping...
Requesting observation 0011420201
found directory "/home/manu/data/trapallada/0011420201", skipping...
Requesting observation 0011430201
found directory "/home/manu/data/trapallada/0011430201", skipping...
Requesting observation 0011450201
found directory "/home/manu/data/trapallada/0011450201", skipping...
Requesting observation 0011830201
found directory "/home/manu/data/trapal

Notice that only files whose name matches the regular expression below are kept.

In [8]:
trapallada.xsa.access.re_filename

re.compile(r'^P(\d{10})PN(S\d{3})(SRCTSR)([0-9A-F]+)\.\w+$', re.UNICODE)

So that, e.g.,

In [9]:
trapallada.xsa.access.re_filename.match('P0006010301PNS003SRSPEC0007.FTZ')

is kept but not

In [10]:
trapallada.xsa.access.re_filename.match('XMM_OM_0760_0001_NIXTMS.FIT')

Let's take a look at the files in the last observation

In [11]:
obs_dir = output_directory / obs_id
obs_files = list(obs_dir.glob('*.FTZ'))
obs_files

[PosixPath('/home/manu/data/trapallada/0011830201/P0011830201PNS003SRCTSR8035.FTZ'),
 PosixPath('/home/manu/data/trapallada/0011830201/P0011830201PNS003SRCTSR801B.FTZ'),
 PosixPath('/home/manu/data/trapallada/0011830201/P0011830201PNS003SRCTSR800B.FTZ'),
 PosixPath('/home/manu/data/trapallada/0011830201/P0011830201PNX000SRCTSR800D.FTZ'),
 PosixPath('/home/manu/data/trapallada/0011830201/P0011830201PNS003SRCTSR801C.FTZ'),
 PosixPath('/home/manu/data/trapallada/0011830201/P0011830201PNX000SRCTSR8035.FTZ'),
 PosixPath('/home/manu/data/trapallada/0011830201/P0011830201PNX000SRCTSR8005.FTZ'),
 PosixPath('/home/manu/data/trapallada/0011830201/P0011830201PNS003SRCTSR8013.FTZ'),
 PosixPath('/home/manu/data/trapallada/0011830201/P0011830201PNS003SRCTSR8028.FTZ'),
 PosixPath('/home/manu/data/trapallada/0011830201/P0011830201PNX000SRCTSR8002.FTZ'),
 PosixPath('/home/manu/data/trapallada/0011830201/P0011830201PNX000SRCTSR8023.FTZ'),
 PosixPath('/home/manu/data/trapallada/0011830201/P0011830201PNX0

# Low-level calls to `XMMNewton`

## `v_instrument_mode` table

*Instrument mode* table

In [12]:
instrument_mode_table = 'v_instrument_mode'
XMMNewton.get_columns('xsa.' + instrument_mode_table)

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]


['instrument', 'instrument_mode_oid', 'mode', 'mode_friendly_name']

A sample

In [13]:
table = XMMNewton.query_xsa_tap(
    f'select TOP 10 * from {instrument_mode_table}',
)
table.to_pandas()

,instrument,instrument_mode_oid,mode,mode_friendly_name
0,EMOS1,27,CcdDiagnostic,Diagnostic 1x1
1,EMOS1,28,Diagnostic1x1,Diagnostic 1x1
2,EMOS1,29,Diagnostic3x3,Diagnostic 3x3
3,EMOS1,30,FastCompressed,Timing Compressed
4,EMOS1,31,FastUncompressed,Timing
5,EMOS1,32,OffsetVariance,Offset Var.
6,EMOS1,33,PrimeFullWindow,Full Frame
7,EMOS1,34,PrimePartialRFS,Refresh Frame Store
8,EMOS1,35,PrimePartialW2,Small Window
9,EMOS1,36,PrimePartialW3,Large Window
